[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/latincy/latincy-book/blob/main/quickstart.ipynb)

This quickstart introduces LatinCy through a passage from the Latin Bible (Vulgate). In a few minutes you will install the library, load a model, annotate a text, and run a lemma-based search to find all grammatical forms of a Latin word across a passage — useful for close reading, concordance work, and textual analysis.

The examples use the small model (`la_core_web_sm`) and John 1:1–5 as a working text. No prior NLP experience is assumed.

## Setup

The cell below installs spaCy and the LatinCy small model automatically when run in Google Colab. If you are working in a local environment, see the [Installing LatinCy models](2_install.ipynb) chapter for installation instructions.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -U spacy -q
    !pip install -q "la-core-web-sm @ https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.4-py3-none-any.whl"

In [ ]:
import spacy

nlp = spacy.load('la_core_web_sm')
print(f"Loaded pipeline: {nlp.meta['name']} v{nlp.meta['version']}")

## Working with Latin text

We will work with the opening verses of the Gospel of John in the Vulgate (John 1:1–5), a passage familiar to students of Latin theology and patristics:

> *In principio erat Verbum, et Verbum erat apud Deum, et Deus erat Verbum.*

Passing the text to `nlp()` runs it through the full LatinCy pipeline: tokenization, lemmatization, part-of-speech tagging, morphological analysis, dependency parsing, and named entity recognition.

In [ ]:
text = (
    "In principio erat Verbum, et Verbum erat apud Deum, et Deus erat Verbum. "
    "Hoc erat in principio apud Deum. "
    "Omnia per ipsum facta sunt, et sine ipso factum est nihil quod factum est. "
    "In ipso vita erat, et vita erat lux hominum; "
    "et lux in tenebris lucet, et tenebrae eam non comprehenderunt."
)

doc = nlp(text)
print(doc)

## Token annotations

Once a text is processed, every token in the `Doc` carries a set of linguistic annotations. The most commonly used are:

| Attribute | Description |
|-----------|-------------|
| `token.text` | The original surface form |
| `token.lemma_` | The dictionary headword |
| `token.pos_` | Coarse part-of-speech tag (`NOUN`, `VERB`, `ADP`, …) |
| `token.morph` | Full morphological feature bundle |

The first sentence of our passage illustrates the range of annotation across a short Latin text:

In [ ]:
first_sent = list(doc.sents)[0]

print(f"{'Token':<14} {'Lemma':<14} {'POS':<8} Morphology")
print("-" * 64)
for token in first_sent:
    if not token.is_punct and not token.is_space:
        print(f"{token.text:<14} {token.lemma_:<14} {token.pos_:<8} {token.morph}")

## Finding words with the Matcher

One of the most useful things a Latin NLP pipeline can do is find all grammatical forms of a word in a passage — the task a concordance handles, but with the model doing the morphological work rather than a pre-built index.

spaCy's `Matcher` lets you search by annotation attributes. Matching on `LEMMA` finds every surface form the model has traced back to a given headword. For the Johannine Prologue, this lets you locate every occurrence of *deus* in all its declined forms:

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
matcher.add('DEUS', [[{'LEMMA': 'deus'}]])

matches = matcher(doc)
print(f"{'Form':<12} {'POS':<8} Morphology")
print("-" * 52)
for match_id, start, end in matches:
    token = doc[start]
    print(f"{token.text:<12} {token.pos_:<8} {token.morph}")

The same approach scales to a set of theological key terms. Adding patterns for *verbum*, *vita*, and *lux* builds a simple concordance for the Prologue's central vocabulary:

In [ ]:
matcher = Matcher(nlp.vocab)
key_terms = ['deus', 'verbum', 'vita', 'lux']
for term in key_terms:
    matcher.add(term.upper(), [[{'LEMMA': term}]])

matches = matcher(doc)
print(f"{'Lemma':<10} {'Form':<14} Morphology")
print("-" * 56)
for match_id, start, end in matches:
    token = doc[start]
    label = nlp.vocab.strings[match_id].lower()
    print(f"{label:<10} {token.text:<14} {token.morph}")

## Next steps

This quickstart covers the basics. The full book works through each pipeline component in detail:

- **[Installing LatinCy models](2_install.ipynb)** — all four model sizes and optional packages
- **[Key annotations](4_key-annotations.ipynb)** — complete reference for token, span, and doc attributes
- **[Lemmatization](lemmatization.ipynb)** — how the lemmatizer works and its edge cases
- **[Sequence Matching](matcher.ipynb)** — the full `Matcher` API with operators, quantifiers, and regex patterns
- **[Named Entity Recognition](ner.ipynb)** — finding people, places, and other entities in Latin texts